In [1]:
!git clone https://github.com/jkelle11-source/msml612_project.git
%cd msml612_project
!git checkout m3-2_sankha
!uv sync --locked
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"

Cloning into 'msml612_project'...
remote: Enumerating objects: 416, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 416 (delta 84), reused 108 (delta 69), pack-reused 258 (from 1)
Receiving objects: 100% (416/416), 387.57 KiB | 11.07 MiB/s, done.
Resolving deltas: 100% (220/220), done.
/content/msml612_project
branch 'm3-2_sankha' set up to track 'origin/m3-2_sankha'.
Switched to a new branch 'm3-2_sankha'
Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 67 packages in 31ms
Prepared 63 packages in 55.59s
Installed 63 packages in 521ms
 + annotated-types==0.8.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + einops==0.8.2
 + filelock==4.0.12
 + fsspec==2026.9.0
 + googleapis-common-protos==1.75.5
 + idna==3.20
 + iniconfig==2.3.1
 + jinja2==3.1.6
 + markupsafe==3.0.4
 

In [2]:
!git fetch origin && git checkout m3-2_sankha && git pull
!uv sync

Already on 'm3-2_sankha'
Your branch is up to date with 'origin/m3-2_sankha'.
Already up to date.
Resolved 67 packages in 1ms
Checked 63 packages in 0.77ms


In [3]:
!mkdir -p data/raw && wget -qO- "https://physionet.org/files/challenge-2012/1.0.0/set-a.tar.gz?download" | tar -xz -C data/raw
!ls data/raw/set-a | wc -l
!uv run python -m data.physionet

4000


In [4]:
from google.colab import userdata
import os; os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
!uv run python -m diffusion.run --seed 42 --epochs 3 --validate-every 3 --wandb disabled --out outputs/m3-2_check_untracked
!uv run python -m diffusion.run --seed 42 --epochs 3 --validate-every 3 --tag check --out outputs/m3-2_check_tracked

base seed 42 on cuda, 3 epochs -> outputs/m3-2_check_untracked
/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)
epoch 1/3  loss 0.3985  max grad norm 7.56  [32s]
epoch 2/3  loss 0.3355  max grad norm 1.91  [26s]
epoch 3/3  loss 0.3272  max grad norm 1.20  val MAE 0.4850  val RMSE 0.7643  [val 300s]  (forward fill 0.4366)  [26s]
{
 "variant": "base",
 "seed": 42,
 "device": "cuda",
 "validation": {
  "split": "val",
  "n_samples": 5,
  "batch_size": 64
 },
 "health": {
  "n_steps": 500,
  "finite": true,
  "loss_first_tenth": 0.5065789216756821,
  "loss_last_tenth": 0.32290125846862794,
  "grad_norm_median": 0.6766524612903595,
  "grad_norm_max": 7.555301189422607,
  "grad_norm_max_step": 16,
  "spikes": [
   {
    "step": 16,
    "l

In [5]:
!cmp outputs/m3-2_check_untracked/history.jsonl outputs/m3-2_check_tracked/history.jsonl && echo "history identical"
!uv run python -c "import torch; a, b = (torch.load(f'outputs/m3-2_check_{d}/model.pt', map_location='cpu') for d in ('untracked', 'tracked')); print('weights identical:', a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a))"

cmp: outputs/m3-2_check_tracked/history.jsonl: No such file or directory
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import torch; a, b = (torch.load(f'outputs/m3-2_check_{d}/model.pt', map_location='cpu') for d in ('untracked', 'tracked')); print('weights identical:', a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a))
                  ^^^^
  File "<string>", line 1, in <genexpr>
    import torch; a, b = (torch.load(f'outputs/m3-2_check_{d}/model.pt', map_location='cpu') for d in ('untracked', 'tracked')); print('weights identical:', a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a))
                          ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/msml612_project/.venv/lib/python3.13/site-packages/torch/serialization.py", line 1565, in load
    with _open_file_like(f, "rb") as opened_file:
         ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/content/msml612_project/.venv/lib/python3.13/s

In [8]:
import os; os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [9]:
!uv run python -c "import wandb; api = wandb.Api(); v = api.viewer; print('user:', v.username, '| default entity:', api.default_entity, '| teams:', v.teams)"

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
user: jakekeller864 | default entity: jkelle11-university-of-maryland | teams: ['jkelle11-university-of-maryland', 'DATA612']


In [10]:
!uv run python -m diffusion.run --seed 42 --epochs 3 --validate-every 3 --tag check --out outputs/m3-2_check_tracked

base seed 42 on cuda, 3 epochs -> outputs/m3-2_check_tracked
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jakekeller864 (jkelle11-university-of-maryland) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run 85b9o6ls (0.0s)
wandb: ⣻ setting up run 85b9o6ls (0.0s)
wandb: ⣽ setting up run 85b9o6ls (0.0s)
wandb: ⣾ setting up run 85b9o6ls (0.0s)
wandb: ⣷ setting up run 85b9o6ls (0.5s)
wandb: ⣯ setting up run 85b9o6ls (0.5s)
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /content/msml612_project/wandb/run-20261022_194446-85b9o6ls
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run base_random_seed42
wandb: ⭐️ View project at https://wandb.ai/DATA612/msml612-csdi
wandb: 🚀 View run at https://wandb.ai/DATA612/msml612-csdi/runs/85b9o6ls
/content/msml612_project/models/csdi/diff_models.py:11: UserWarning: enable_nested_tensor is True,

In [11]:
!cmp outputs/m3-2_check_untracked/history.jsonl outputs/m3-2_check_tracked/history.jsonl && echo "history identical"
!uv run python -c "import torch; a, b = (torch.load(f'outputs/m3-2_check_{d}/model.pt', map_location='cpu') for d in ('untracked', 'tracked')); print('weights identical:', a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a))"

history identical
weights identical: True
